# Скоринг финансовой устойчивости

Тетрадь демонстрирует полный пайплайн: загрузка данных, вычисление финансовых
коэффициентов, обучение LightGBM-классификатора и интерпретация с помощью SHAP.

In [ ]:
import sys
import os
import warnings

warnings.filterwarnings('ignore')

# Добавляем корень проекта в путь
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.collector import load_fallback_data
from src.features import compute_ratios
from src.validator import validate_raw, validate_features
from src.model import FEATURE_COLS, train, load_model, score
from src.explain import compute_shap, waterfall_fig, summary_fig

print('Зависимости загружены')

In [ ]:
# Загрузка данных
# Если доступен DuckDB с собранными данными - читаем его
db_path = '../data/raw.duckdb'

if os.path.exists(db_path):
    import duckdb
    con = duckdb.connect(db_path, read_only=True)
    df_raw = pl.from_pandas(con.execute('SELECT * FROM raw').df())
    con.close()
    print(f'Загружено из DuckDB: {len(df_raw)} записей')
else:
    print('DuckDB не найден, используется резервный датасет')
    df_raw = load_fallback_data()
    print(f'Загружено {len(df_raw)} записей')

# Валидация сырых данных
df_raw = validate_raw(df_raw)
print(f'После валидации: {len(df_raw)} записей')
df_raw.head(5)

In [ ]:
# Вычисление финансовых коэффициентов
df = compute_ratios(df_raw)
df = validate_features(df)

print('Статистика по коэффициентам:')
ratio_cols = ['z_score', 'debt_ratio', 'roa', 'asset_turnover',
              'working_capital_ratio', 'retained_earnings_ratio']
print(df.select(ratio_cols).describe())

# Распределение debt_ratio
fig = make_subplots(rows=2, cols=3, subplot_titles=ratio_cols)
positions = [(1,1),(1,2),(1,3),(2,1),(2,2),(2,3)]

for col, (row, col_pos) in zip(ratio_cols, positions):
    vals = df[col].drop_nulls().to_list()
    # Обрезаем выбросы по 1-99 перцентили для читаемости
    p1, p99 = np.percentile(vals, [1, 99])
    clipped = [v for v in vals if p1 <= v <= p99]
    fig.add_trace(
        go.Histogram(x=clipped, name=col, showlegend=False, nbinsx=40,
                     marker_color='rgba(13,110,253,0.6)'),
        row=row, col=col_pos
    )

fig.update_layout(
    title='Распределение финансовых коэффициентов',
    height=500,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig.show()

In [ ]:
# Распределение Z-score Альтмана с зонами безопасности
z_vals = df['z_score'].drop_nulls().to_list()
p1, p99 = np.percentile(z_vals, [1, 99])
z_clipped = [v for v in z_vals if p1 <= v <= p99]

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=z_clipped, nbinsx=60,
    marker_color='rgba(13,110,253,0.65)',
    name='Z-score'
))

# Вертикальные линии порогов Альтмана
for x_val, label, color in [
    (1.23, 'Зона дефолта (1.23)', 'red'),
    (2.90, 'Безопасная зона (2.90)', 'green'),
]:
    fig.add_vline(
        x=x_val, line_dash='dash', line_color=color, line_width=2,
        annotation_text=label, annotation_position='top right'
    )

default_share = df.filter(pl.col('z_score') < 1.23).shape[0] / len(df) * 100
print(f'Доля компаний в зоне дефолта (Z < 1.23): {default_share:.1f}%')

fig.update_layout(
    title='Распределение Z-score Альтмана',
    xaxis_title='Z-score',
    yaxis_title='Количество компаний',
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=400
)
fig.show()

In [ ]:
# Обучение LightGBM-модели
import os

model_path = '../models/lgbm.pkl'
os.makedirs('../models', exist_ok=True)

train(df, model_path)

model = load_model(model_path)
df_scored = score(df, model)

print('\nРаспределение по риск-классам:')
print(df_scored.group_by('risk_label').agg(pl.count().alias('count')).sort('count', descending=True))

In [ ]:
# SHAP summary plot - важность признаков по всей выборке
shap_values, explainer = compute_shap(df_scored, model)

feature_cols = model['feature_cols']
fig_summary = summary_fig(shap_values, feature_cols)
fig_summary.show()

# Waterfall для первой компании с высоким риском
high_risk = df_scored.filter(pl.col('risk_label') == 'Высокий')
if len(high_risk) > 0:
    inn_all = df_scored['inn'].to_list()
    target_inn = high_risk['inn'][0]
    idx = inn_all.index(target_inn)
    fig_wf = waterfall_fig(shap_values, idx, feature_cols)
    fig_wf.update_layout(title=f'SHAP waterfall: ИНН {target_inn} (Высокий риск)')
    fig_wf.show()
else:
    print('Компаний с высоким риском не найдено')

## Выводы

**Z-score Альтмана** - аналитический метод, основанный на фиксированных весах
коэффициентов, выведенных на данных американских компаний 1960-х годов.
Его главное преимущество - полная интерпретируемость и отсутствие необходимости
в размеченных данных. Основные недостатки: чувствительность к отраслевой специфике,
устаревшие пороговые значения, неустойчивость при нестандартной структуре баланса.

**LightGBM** обучается непосредственно на исторических данных с известным исходом
(дефолт / не дефолт), что позволяет ему учитывать нелинейные зависимости
и взаимодействия между признаками. Модель показывает более высокую
дискриминирующую способность (AUC-ROC), особенно на несбалансированных выборках.
Недостаток - требует размеченной обучающей выборки и регулярного переобучения
при смещении распределения данных.

**SHAP waterfall** устраняет главный недостаток ML-моделей - непрозрачность
решений. Для каждой компании аналитик видит, какие именно коэффициенты
повышают или снижают вероятность дефолта, что делает скоринг пригодным
для кредитного анализа и регуляторной отчетности.

Рекомендуемый подход: использовать Z-score как первичный быстрый фильтр,
LightGBM - для точного ранжирования пограничных случаев (1.23 < Z < 2.9),
SHAP - для объяснения решений аналитикам и клиентам.